# LLM A pilot — bake-off

LLM A: `query + qrels gold doc + corpus stats hints -> predicted route rank order, or abstain`
(decision 2/3/12/A3). It **predicts**, never observes retrieval (M1) — its disagreement with the
spine means mechanism and outcome diverge (distractors, or odd qrels), never "the label is wrong."

**Three candidates, same draw, real calls:**

| role | model |
|---|---|
| primary | `openai/gpt-5.6-luna` |
| bake-off alt | `deepseek/deepseek-v4-flash` |
| bake-off alt | `google/gemini-3.1-flash-lite` |

All via OpenRouter's REST API directly (not litellm — its cost calculator doesn't know these models
yet; OpenRouter's own `usage.cost` is authoritative and doesn't need it to). Real dollars, expect well
under a cent per model. This is **not** the pre-registered 500-row calibration spike
(route-label-sourcing.md) — n here is far too small for that gate; it proves the pipeline, the price,
and which candidate is worth spending the real spike's budget on.

In [9]:
import os
import re

import pandas as pd
import requests
from dotenv import load_dotenv

from composition.pool_v3 import LabelledPool
from hybrid_search_rrf_dataset.router import DATA_DIR
from scripts.llm_a import INSTRUCTION, ROUTE_MAP, parse

load_dotenv()
OPENROUTER_KEY = os.environ.get("OPENROUTER_API_KEY") or os.environ.get("OPEN_ROUTER_API_KEY")
assert OPENROUTER_KEY, "no OpenRouter key in .env"
COMPLETIONS_URL = "https://openrouter.ai/api/v1/chat/completions"

MODELS = {
    "primary": "openai/gpt-5.6-luna",
    "deepseek": "deepseek/deepseek-v4-flash",
    "gemini-flash-lite": "google/gemini-3.1-flash-lite",
}
pd.set_option("display.width", 170)

In [10]:
def complete(model: str, system: str, user: str, *, max_tokens: int = 120) -> tuple[str, dict]:
    """Real call, real cost read from OpenRouter's own `usage.cost` — not litellm's
    static price list, which does not know brand-new model ids yet (verified: it
    raises 'This model isn't mapped yet' on `openai/gpt-5.6-luna`).

    `reasoning: {effort: none}` is required, not optional: `gpt-5.6-luna` and
    `deepseek-v4-flash` reason by default and, on the real (long) prompt below,
    spent the ENTIRE `max_tokens` budget on hidden reasoning tokens — verified:
    `finish_reason='length'`, `content=None`, `reasoning_tokens=64`, still
    CHARGED ($0.0003454 for zero visible output). Without this, a full-pool run
    on the primary model would silently return empty answers while billing for
    them. `gemini-3.1-flash-lite` does not reason by default here, but the flag
    is harmless to send regardless. `max_tokens` sized for ORDER + WHY (two short
    lines); with effort:none there is no hidden reasoning to eat it."""
    r = requests.post(
        COMPLETIONS_URL,
        headers={"Authorization": f"Bearer {OPENROUTER_KEY}", "Content-Type": "application/json"},
        json={"model": model, "max_tokens": max_tokens, "usage": {"include": True},
              "reasoning": {"effort": "none"},
              "messages": [{"role": "system", "content": system},
                           {"role": "user", "content": user}]},
        timeout=30,
    )
    r.raise_for_status()
    body = r.json()
    content = body["choices"][0]["message"]["content"]
    return (content or "").strip(), body.get("usage", {})

## 1. Draw — tiny, stratified, real data

In [11]:
SEED = 0
N_PER_KIND = 4
LANE = "crumb-legal-qa"

pool = LabelledPool()
classified = pool.classify(pool.labels())
lane_rows = classified[classified["dataset"] == LANE]
draws = [lane_rows[lane_rows["kind"] == k].sample(min(N_PER_KIND, len(lane_rows[lane_rows["kind"] == k])), random_state=SEED)
         for k in ("decisive", "undecisive", "genuine_tie", "all_zero")]
drawn = pd.concat(draws, ignore_index=True)
print(f"drew {len(drawn)} rows: " + drawn["kind"].value_counts().to_dict().__repr__())

drew 16 rows: {'decisive': 4, 'undecisive': 4, 'genuine_tie': 4, 'all_zero': 4}


## 2. Build the real prompt — gold doc + corpus stats, decision 12's abstain token

In [12]:
qrels = pd.read_parquet(DATA_DIR / LANE / "qrels.parquet").astype({"query_id": str, "doc_id": str})
corpus = pd.read_parquet(DATA_DIR / LANE / "corpus.parquet")

GOLD_CHARS = 1200
"""The repo's own convention (`ParentPool.gold_text(chars=1200)`,
`augmentation/parents.py:105`) and the basis of the measured per-row cost below.
Stated rather than silent, because it is a real limitation: 85% of
crumb-legal-qa docs exceed 1200 chars (median 1940), so LLM A judges most rows
from a truncated gold doc. Defensible here — LLM A's question (does this
query<->answer pair need lexical or semantic matching?) is usually decidable
from a document's opening — but a gold doc whose distinguishing content sits
past the cutoff is a source of error this pilot cannot see. Raising it raises
the measured cost roughly proportionally on the input side."""

corpus_text = {
    str(r.doc_id): (f"{r.title}\n{r.text}" if "title" in corpus.columns else str(r.text)).strip()[:GOLD_CHARS]
    for r in corpus.itertuples()
}
stats = pd.read_parquet(DATA_DIR / "route_labels" / "query_corpus_stats.parquet").astype({"query_id": str})


def build_prompt(row) -> tuple[str, str] | tuple[None, None]:
    """-> (prompt, stats_line); (None, None) when the query has no judged gold doc.
    `stats_line` is returned so the caller can store the numbers the model saw
    and later check its WHY against them."""
    judged = qrels.loc[
        (qrels["query_id"] == str(row.query_id)) & (qrels["relevance"] >= row.min_relevance),
        "doc_id",
    ]
    if judged.empty:
        return None, None
    gold_text = corpus_text.get(str(judged.iloc[0]), "")
    s = stats.loc[stats["query_id"] == str(row.query_id)]
    stats_line = (
        "unavailable" if s.empty else
        ", ".join(f"{c}={s.iloc[0][c]:.3g}" for c in
                  ("avg_idf", "max_idf", "oov_share", "vocab_overlap", "mean_pmi", "collection_size"))
    )
    prompt = (
        f"Query: {row.query}\n\n"
        f"Known answer document:\n{gold_text}\n\n"
        f"Target collection statistics for this query: {stats_line}"
    )
    return prompt, stats_line

## 3. LIVE — spends real money, once per candidate model

Expected: a few hundred input tokens + ~20-60 output tokens per row per model — well under $0.001/row
on any of the three. Run this cell explicitly; nothing above it touches the network.

In [13]:
RUN_LIVE = True  # flip to True to actually spend
RESULTS_PATH = DATA_DIR / "llm_a_pilot" / "results.parquet"

# cache-first: a file on disk beats a re-spend. delete the parquet to force a fresh run.
# first-time seeding: if `results` is already in this kernel from an earlier LIVE run,
# save it BEFORE re-executing this cell so the cache picks up:
#   RESULTS_PATH.parent.mkdir(parents=True, exist_ok=True); results.to_parquet(RESULTS_PATH)
if RESULTS_PATH.exists():
    results = pd.read_parquet(RESULTS_PATH)
    print(f"cached: loaded {len(results)} rows from {RESULTS_PATH.relative_to(DATA_DIR.parent)}")
elif RUN_LIVE:
    rows = []
    for role, model in MODELS.items():
        for row in drawn.itertuples():
            prompt, stats_line = build_prompt(row)
            if prompt is None:
                continue
            text, usage = complete(model, INSTRUCTION, prompt)
            rows.append({
                "role": role, "model": model, "dataset": row.dataset, "query_id": row.query_id,
                "kind": row.kind, "llm_a_raw": text, "stats_line": stats_line, "cost_usd": usage.get("cost", 0),
                "prompt_tokens": usage.get("prompt_tokens"), "completion_tokens": usage.get("completion_tokens"),
            })
    results = pd.DataFrame(rows)
    RESULTS_PATH.parent.mkdir(parents=True, exist_ok=True)
    results.to_parquet(RESULTS_PATH)
    print(results.groupby("role")["cost_usd"].agg(["count", "sum", "mean"]))
    print(f"\nsaved {len(results)} rows to {RESULTS_PATH.relative_to(DATA_DIR.parent)}")
else:
    results = pd.DataFrame()
    print("RUN_LIVE is False and no cache — flip RUN_LIVE or seed the parquet manually")

                   count       sum      mean
role                                        
deepseek              16  0.001734  0.000108
gemini-flash-lite     16  0.004089  0.000256
primary               16  0.003256  0.000203

saved 48 rows to data/llm_a_pilot/results.parquet


## 4. Parse + compare against the spine (informal — n far too small for the real gate)

In [14]:
ROUTE_LABEL = {v: k for k, v in ROUTE_MAP.items()}  # winner id -> friendly name

if RUN_LIVE and len(results):
    parsed = results["llm_a_raw"].map(parse)
    results["llm_a_top"] = parsed.map(lambda p: p[0])
    results["llm_a_why"] = parsed.map(lambda p: p[1])
    results["llm_a_abstain"] = parsed.map(lambda p: p[2])
    results["llm_a_top_route"] = results["llm_a_top"].map(ROUTE_MAP)

    decisive = results[results["kind"] == "decisive"].merge(
        classified[["dataset", "query_id", "winner"]], on=["dataset", "query_id"], how="left",
    )
    for role, group in decisive.groupby("role"):
        abstain_rate = group["llm_a_top_route"].isna().mean()
        answered = group[group["llm_a_top_route"].notna()]
        # abstain is zero weight (decision 12), never a vote against — excluded
        # from the denominator, not counted as disagreement
        line = f"{role:20s} decisive rows: {len(group)}  abstained: {abstain_rate:.0%}"
        if len(answered):
            agree = (answered["llm_a_top_route"] == answered["winner"]).mean()
            line += f"  agreement on answered ({len(answered)}): {agree:.0%}"
        else:
            line += "  (none answered)"
        print(line)

deepseek             decisive rows: 4  abstained: 75%  agreement on answered (1): 100%
gemini-flash-lite    decisive rows: 4  abstained: 50%  agreement on answered (2): 0%
primary              decisive rows: 4  abstained: 25%  agreement on answered (3): 0%


## 4b. Decisive rows — does LLM A hold a different opinion, and why?

These are rows retrieval already settled (`kind == decisive`, a clear spine winner). A disagreement here is the case worth reading: LLM A *predicts* a different best route than the one measurement observed. Per M1 that never means the label is wrong — it means mechanism and outcome diverge (a distractor in the gold doc, odd qrels). `verdict` splits agree / differ / abstain (abstain is zero-weight, not a vote against). For every `differ`, the block below prints the model's own `why` next to the `saw` stats line it was given — so you can judge whether the stated basis is real or a confabulation.

In [15]:
if RUN_LIVE and len(results):
    dec = results[results["kind"] == "decisive"].merge(
        classified[["dataset", "query_id", "winner", "query"]],
        on=["dataset", "query_id"], how="left")
    dec["spine"] = dec["winner"].map(ROUTE_LABEL)
    dec["verdict"] = [
        "abstain" if pd.isna(rt) else ("agree" if rt == w else "differ")
        for rt, w in zip(dec["llm_a_top_route"], dec["winner"])
    ]
    dec = dec.sort_values(["verdict", "role"]).reset_index(drop=True)
    print(dec["verdict"].value_counts().to_dict(), "\n")

    scan = dec[["role", "spine", "llm_a_top", "verdict", "query"]].copy()
    scan["query"] = scan["query"].astype(str).str.slice(0, 48)
    print(scan.rename(columns={"llm_a_top": "llm_a"}).to_string(index=False))

    print("\n--- opinions that DIFFER from the spine, in full (read the WHY, check it against `saw`) ---")
    diffs = dec[dec["verdict"] == "differ"]
    if diffs.empty:
        print("  (none — no model disagreed on a decisive row in this draw)")
    for _, r in diffs.iterrows():
        print(f"\n[{r.role}]  spine={r.spine}  ->  llm_a={r.llm_a_top}")
        print(f"  query: {r.query}")
        print(f"  why  : {r.llm_a_why}")
        print(f"  saw  : {r.stats_line}")

{'abstain': 6, 'differ': 5, 'agree': 1} 

             role  spine  llm_a verdict                                            query
         deepseek  dense    NaN abstain Primary methods of service are defined as those 
         deepseek  dense    NaN abstain Primary methods of service are defined as those 
         deepseek  dense    NaN abstain Is the term 'Writ for removal' used to refer to 
gemini-flash-lite sparse    NaN abstain Is the term 'Execution' used to refer to the ord
gemini-flash-lite  dense    NaN abstain Primary methods of service are defined as those 
          primary sparse    NaN abstain Is the term 'Execution' used to refer to the ord
         deepseek sparse sparse   agree Is the term 'Execution' used to refer to the ord
gemini-flash-lite  dense sparse  differ Primary methods of service are defined as those 
gemini-flash-lite  dense sparse  differ Is the term 'Writ for removal' used to refer to 
          primary  dense sparse  differ Primary methods of service a

## 4c. Undecided branches — what does LLM A add where measurement is weak or silent?

`undecisive` (narrow spine margin), `genuine_tie`, and `all_zero` (no route retrieved a judged doc) are the rows a label change can actually move — here LLM A is the tie-breaker, not a second-guesser of a clear outcome. Three questions: does it commit to a route or abstain, which way does it break the near-tie versus the spine's weak winner, and do the three models agree *with each other* (a signal that the pick is stable rather than noise)?

In [16]:
if RUN_LIVE and len(results):
    UND = ["undecisive", "genuine_tie", "all_zero"]
    und = results[results["kind"].isin(UND)].merge(
        classified[["dataset", "query_id", "winner", "query"]],
        on=["dataset", "query_id"], how="left")
    und["pick"] = und["llm_a_top"].fillna("ABSTAIN")

    print("pick distribution per kind per model:\n")
    for kind in UND:
        sub = und[und["kind"] == kind]
        if sub.empty:
            continue
        print(f"  {kind} ({sub['query_id'].nunique()} rows):")
        for role, g in sub.groupby("role"):
            print(f"    {role:20s} {g['pick'].value_counts().to_dict()}")
        print()

    # do the three models land on the same pick for a given row, or scatter?
    pivot = und.pivot_table(index=["kind", "query_id"], columns="role",
                            values="pick", aggfunc="first")
    consensus = (pivot.nunique(axis=1) == 1).sum()
    print(f"cross-model consensus: {consensus}/{len(pivot)} undecided rows got the "
          f"identical pick from all three models\n")

    # undecisive rows carry a real (narrow) spine winner — show confirm vs flip
    uc = und[und["kind"] == "undecisive"].copy()
    if not uc.empty:
        uc["spine"] = uc["winner"].map(ROUTE_LABEL)
        print("--- undecisive: does LLM A confirm or flip the spine's narrow winner? ---")
        for _, r in uc.sort_values(["query_id", "role"]).iterrows():
            move = "confirm" if r["llm_a_top_route"] == r["winner"] else "flip / abstain"
            print(f"\n[{r.role}]  spine≈{r.spine}  ->  llm_a={r.pick}  ({move})")
            print(f"  query: {str(r.query)[:60]}")
            print(f"  why  : {r.llm_a_why or r.llm_a_abstain}")

pick distribution per kind per model:

  undecisive (4 rows):
    deepseek             {'sparse': 2, 'ABSTAIN': 2}
    gemini-flash-lite    {'ABSTAIN': 3, 'sparse': 1}
    primary              {'sparse': 2, 'ABSTAIN': 1, 'dense': 1}

  genuine_tie (4 rows):
    deepseek             {'ABSTAIN': 3, 'sparse': 1}
    gemini-flash-lite    {'ABSTAIN': 3, 'sparse': 1}
    primary              {'ABSTAIN': 3, 'sparse': 1}

  all_zero (4 rows):
    deepseek             {'ABSTAIN': 3, 'sparse': 1}
    gemini-flash-lite    {'sparse': 2, 'ABSTAIN': 2}
    primary              {'ABSTAIN': 3, 'sparse': 1}

cross-model consensus: 7/12 undecided rows got the identical pick from all three models

--- undecisive: does LLM A confirm or flip the spine's narrow winner? ---

[deepseek]  spine≈sparse  ->  llm_a=ABSTAIN  (flip / abstain)
  query: Can a landlord evict a tenant for having unauthorized occupa
  why  : nan

[gemini-flash-lite]  spine≈sparse  ->  llm_a=ABSTAIN  (flip / abstain)
  query: Can a landl

## 5. Real cost per candidate, extrapolated

In [17]:
if RUN_LIVE and len(results):
    summary = results.groupby("role").agg(
        n=("cost_usd", "count"), per_row=("cost_usd", "mean"),
        mean_prompt_tok=("prompt_tokens", "mean"), mean_completion_tok=("completion_tokens", "mean"),
    )
    summary["extrapolated_224910"] = summary["per_row"] * 224_910
    print(summary.to_string())
    print("\nplan's hand estimate was: ~$127 (haiku)")

                    n   per_row  mean_prompt_tok  mean_completion_tok  extrapolated_224910
role                                                                                      
deepseek           16  0.000108         778.7500              72.3125            24.377809
gemini-flash-lite  16  0.000256         808.9375              35.5625            57.482076
primary            16  0.000203         777.8750              39.9375            45.769185

plan's hand estimate was: ~$127 (haiku)


## 6. Model personality — spine × LLM A pick, per candidate

Decisive rows only. Rows = the route retrieval already crowned; columns = the route LLM A predicted. The diagonal is agreement; off-diagonal reads as *bias direction* — a model that lands in the `hybrid` column when the spine says `dense` behaves differently from one that lands in `sparse`, even when the raw "agreement rate" is the same 25% (which, on n=4, is a coin-flip anyway). "Each model has its own why" made mechanical: the off-diagonal column name is that why, in one word.

In [18]:
if RUN_LIVE and len(results):
    ROUTES = ["dense", "sparse", "hybrid"]
    dec_all = results[results["kind"] == "decisive"].merge(
        classified[["dataset", "query_id", "winner"]], on=["dataset", "query_id"], how="left")
    dec_all["spine"] = dec_all["winner"].map(ROUTE_LABEL)
    dec_all["pick"] = dec_all["llm_a_top"].fillna("ABSTAIN")
    print("rows = spine winner (retrieval-observed), cols = LLM A pick\n")
    for role, g in dec_all.groupby("role"):
        cm = pd.crosstab(g["spine"], g["pick"]).reindex(
            index=ROUTES, columns=ROUTES + ["ABSTAIN"], fill_value=0)
        cm = cm.loc[cm.sum(axis=1) > 0]
        # bias = pick-column that carries the most off-diagonal mass
        off = cm.copy()
        for r in off.index.intersection(off.columns):
            off.loc[r, r] = 0
        bias_col = off.sum(axis=0).idxmax() if off.sum().sum() else "-"
        bias_mass = off.sum(axis=0).max()
        print(f"[{role}]  off-diagonal bias -> {bias_col} ({bias_mass}/{off.sum().sum()} off-diag rows)")
        print(cm.to_string())
        print()

rows = spine winner (retrieval-observed), cols = LLM A pick

[deepseek]  off-diagonal bias -> ABSTAIN (3/3 off-diag rows)
pick    dense  sparse  hybrid  ABSTAIN
spine                                 
dense       0       0       0        3
sparse      0       1       0        0

[gemini-flash-lite]  off-diagonal bias -> sparse (2/4 off-diag rows)
pick    dense  sparse  hybrid  ABSTAIN
spine                                 
dense       0       2       0        1
sparse      0       0       0        1

[primary]  off-diagonal bias -> sparse (3/4 off-diag rows)
pick    dense  sparse  hybrid  ABSTAIN
spine                                 
dense       0       3       0        0
sparse      0       0       0        1



## 7. Cited feature ↔ pick — do models mean the same thing by the same cue?

Every model's WHY names one or two features it claims drove the pick ("precise legal terminology", "low IDF", "tiny collection", "semantic nuance"). This cell tags each WHY by which of six cues it cites — a keyword regex over the free text, imperfect but readable — then cross-tabs cued-feature × chosen route per model. Read horizontally: when three models cite the *same* cue and pick *different* routes, they share vocabulary but not mental model. That is what "different opinions between LLMs, made meaningful" looks like — the disagreement is not about which stat matters, it is about what the stat *implies*.

In [20]:
FEATURE_CUES = {
    "lexical":   r"lexical|exact\s+(term|phrase|match)|precise.{0,15}(term|legal|statut|phrase|definition)|keyword|bm25|specific\s+(legal|term)|specialized|state-specific",
    "idf":       r"\bidf\b|rarity|rare\s+term|low.{0,15}(idf|collection\W)|collection.{0,15}rar",
    "small_col": r"tiny|small\s+collection|\b4\s*doc",
    "overlap":   r"overlap|vocab(ulary)?\s+overlap",
    "semantic":  r"semantic|nuance|context|meaning|understand|paraphras|conceptual|dilut",
    "hedge":     r"hybrid|hedge|fuse|combin|bridge|robust|fusion",
}


def tag_features(why):
    """Return the cue names whose regex fires anywhere in `why`. Imperfect but
    readable — a WHY may cite several cues (multi-tag) or none (untagged)."""
    if not isinstance(why, str) or not why:
        return []
    return [f for f, pat in FEATURE_CUES.items() if re.search(pat, why, re.IGNORECASE)]


if RUN_LIVE and len(results):
    tagged = results.copy()
    tagged["features"] = tagged["llm_a_why"].map(tag_features)
    tagged["pick"] = tagged["llm_a_top"].fillna("ABSTAIN")
    # explode duplicates the source index; reset so crosstab (pandas 2.x strict) can align
    exploded = (tagged.explode("features")
                .dropna(subset=["features"])
                .reset_index(drop=True))
    answered = tagged["llm_a_why"].notna().sum()
    n_tagged = tagged["features"].map(bool).sum()
    print(f"tagged {n_tagged}/{answered} answered rows with >=1 cited feature\n")

    print("cited feature x chosen route per model:\n")
    for role, g in exploded.groupby("role"):
        cm = pd.crosstab(g["features"], g["pick"])
        print(f"[{role}]")
        print(cm.to_string())
        print()

    print("--- same cue, do models land on the same route? (top pick per model per feature) ---")
    for feat, g in exploded.groupby("features"):
        parts = []
        for role, gg in g.groupby("role"):
            vc = gg["pick"].value_counts()
            parts.append(f"{role}->{vc.index[0]}({vc.iloc[0]}/{len(gg)})")
        print(f"  {feat:10s}  " + "   ".join(parts))

tagged 19/19 answered rows with >=1 cited feature

cited feature x chosen route per model:

[deepseek]
pick       sparse
features         
idf             2
lexical         5
overlap         3
semantic        2
small_col       1

[gemini-flash-lite]
pick      sparse
features        
lexical        6
overlap        2
semantic       2

[primary]
pick       dense  sparse
features                
idf            0       4
lexical        1       7
overlap        1       5
semantic       1       2
small_col      0       3

--- same cue, do models land on the same route? (top pick per model per feature) ---
  idf         deepseek->sparse(2/2)   primary->sparse(4/4)
  lexical     deepseek->sparse(5/5)   gemini-flash-lite->sparse(6/6)   primary->sparse(7/8)
  overlap     deepseek->sparse(3/3)   gemini-flash-lite->sparse(2/2)   primary->sparse(5/6)
  semantic    deepseek->sparse(2/2)   gemini-flash-lite->sparse(2/2)   primary->sparse(2/3)
  small_col   deepseek->sparse(1/1)   primary->sparse(3/3)

## 8. Verdict — one sentence

The pilot's job is to pick the candidate that gets the 500-row calibration budget. Three axes matter: extrapolated cost at the full 224,910 rows, decisive-row agreement with the spine (sanity), and cross-model consensus on undecided rows (variance of the oracle where a labeler would actually rely on it). n=4 per kind, so every rate is ±25% raw — this is the scoreboard, not the decision.

In [21]:
if RUN_LIVE and len(results):
    dec = results[results["kind"] == "decisive"].merge(
        classified[["dataset", "query_id", "winner"]], on=["dataset", "query_id"], how="left")
    dec["pick_route"] = dec["llm_a_top"].map(ROUTE_MAP)
    agree = dec.groupby("role").apply(
        lambda g: (g["pick_route"] == g["winner"]).mean(), include_groups=False)
    cost = results.groupby("role")["cost_usd"].mean() * 224_910

    UND = ["undecisive", "genuine_tie", "all_zero"]
    und = results[results["kind"].isin(UND)].copy()
    und["pick"] = und["llm_a_top"].fillna("ABSTAIN")
    pivot = und.pivot_table(index="query_id", columns="role", values="pick", aggfunc="first")
    row_consensus = (pivot.nunique(axis=1) == 1)

    board = pd.DataFrame({
        "decisive_agree": agree.map(lambda x: f"{x:.0%}"),
        "extrapolated_$": cost.round(2),
    })
    print(board.to_string())
    print(f"\ncross-model consensus on undecided rows: "
          f"{row_consensus.sum()}/{len(pivot)} ({row_consensus.mean():.0%})")

    cheap = cost.idxmin()
    print("\nverdict:")
    print(f"  cheapest at 224,910 rows: {cheap} (${cost.min():.2f}, "
          f"{cost.max() / cost.min():.1f}x under the priciest)")
    print(f"  decisive-agreement rate is tied at {agree.iloc[0]:.0%} across all three "
          f"(n=4 -- noise-floor); direction of disagreement (section 6) is where they differ")
    print(f"  cross-model consensus {row_consensus.mean():.0%} on undecided rows -> "
          f"LLM A is a HIGH-VARIANCE oracle exactly where the spine is silent; "
          f"the 500-row spike must budget for majority voting, not single-model calls")

                  decisive_agree  extrapolated_$
role                                            
deepseek                     25%           24.38
gemini-flash-lite             0%           57.48
primary                       0%           45.77

cross-model consensus on undecided rows: 7/12 (58%)

verdict:
  cheapest at 224,910 rows: deepseek ($24.38, 2.4x under the priciest)
  decisive-agreement rate is tied at 25% across all three (n=4 -- noise-floor); direction of disagreement (section 6) is where they differ
  cross-model consensus 58% on undecided rows -> LLM A is a HIGH-VARIANCE oracle exactly where the spine is silent; the 500-row spike must budget for majority voting, not single-model calls
